In [1]:
from pathlib import Path
import numpy as np
from astropy.io import fits
from PIL import Image



In [ ]:
folder = Path("20260319")
flat_dir = folder / "flat_fits"
browse_dir = folder / "browse"
flat_dir.mkdir(exist_ok=True)
browse_dir.mkdir(exist_ok=True)
for fits_file in folder.glob("*.fits"):
    print(f"Processing {fits_file.name}")
    if "ff" in str(fits_file): 
        continue 
    with fits.open(fits_file) as hdul:
        data = hdul[0].data
    bands, rows, cols = data.shape
    tmp = data.transpose(1, 0, 2)  
    image = tmp
    out_fits = flat_dir / f"{fits_file.stem}_flat.fits"
    fits.writeto(out_fits, image, overwrite=True)

In [ ]:
# ratio of flats to lab flat field 

folder = Path("20260319/ff")
for obs_flat in folder.glob("*.fits"):
    lab  = fits.getdata('20260319/ff/lab_flat_field_global.fits').astype(float)
    obs  = fits.getdata(obs_flat).astype(float)
    
    #product = lab * obs
    #ratio   = obs / lab
    diff = lab-obs
    out_fits = f"{obs_flat.stem}_diff_with_lab.fits"
    fits.writeto(out_fits, diff, overwrite=True)

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
import pandas as pd

df = pd.read_csv("pixels_with_temps.csv")

In [ ]:
df = df[df['obs_id'].str[2] != 't']


In [ ]:
df

In [ ]:
plt.figure(figsize=(10, 10))
# plt.scatter(df['temp'],df['pixel_149_24'],s=.3)
plt.scatter(df['temp'],df['pixel_65_21'],s=.3, label='65, 21')
# plt.scatter(df['temp'],df['pixel_117_63'],s=.3)
# plt.scatter(df['temp'],df['pixel_149_24'],s=.3)
# plt.scatter(df['temp'],df['pixel_219_61'],s=.3)
plt.scatter(df['temp'],df['pixel_209_77'],s=.3, label='209, 77')
plt.scatter(df['temp'],df['pixel_237_44'],s=.3, label='237, 44')
plt.scatter(df['temp'],df['pixel_278_33'],s=.3, label='278, 33')
plt.scatter(df['temp'],df['pixel_284_49'],s=.3, label='284, 49')
plt.scatter(df['temp'],df['pixel_314_9'],s=.3, label='314, 9')
plt.legend()

plt.ylim(400,1600)

In [ ]:
pixel_cols = [col for col in df.columns if col.startswith('pixel_')]

image_std = df.groupby('file')[pixel_cols].apply(lambda x: x.values.flatten().std())

df = df.merge(image_std.rename('pixel_std'), on='file')

In [ ]:
df

In [ ]:
plt.scatter(df['temp'], df['pixel_std'], s=2)
plt.xlabel("temp")
plt.ylabel("std dev between frames of dark")

In [ ]:
import numpy as np

pixel_cols = [col for col in df.columns if col.startswith('pixel_')]

filtered = df[df['frame'] > 1].groupby('file')[pixel_cols]

stats = pd.DataFrame({
    'pixel_std':    filtered.apply(lambda x: np.std(x.values.flatten())),
    'pixel_median': filtered.apply(lambda x: np.median(x.values.flatten())),
    'pixel_mean':   filtered.apply(lambda x: np.mean(x.values.flatten())),
})

df = df.merge(stats, on='file')

In [ ]:
df

In [ ]:
plt.scatter(df['temp'], df['pixel_std_y'], s=2)
plt.xlabel("temp")
plt.ylabel("std dev between frames of dark")

In [ ]:
plt.scatter(df['temp'], df['pixel_mean'], s=2,label='mean')
plt.scatter(df['temp'], df['pixel_median'], s=2,label='median')
plt.legend()
plt.xlabel("temp")
plt.ylabel("dark DN at pixel values")

In [ ]:
pixel_cols = [col for col in df.columns if col.startswith('pixel_')]

filtered = df[df['frame'] > 1].groupby('file')[pixel_cols]

stats = pd.concat([
    filtered.std().add_prefix('std_'),
    filtered.median().add_prefix('median_'),
    filtered.mean().add_prefix('mean_'),
], axis=1)

df = df.merge(stats, on='file')

In [ ]:
df.to_csv("pixel_vals_across_dark_flats.csv")

In [ ]:
df.keys()

In [ ]:
plt.figure(figsize=(10, 10))
plt.scatter(df['temp'],df['std_pixel_149_24'],s=.3)
plt.scatter(df['temp'],df['std_pixel_65_21'],s=.3, label='65, 21')
plt.scatter(df['temp'],df['std_pixel_117_63'],s=.3)
plt.scatter(df['temp'],df['std_pixel_149_24'],s=.3)
plt.scatter(df['temp'],df['std_pixel_219_61'],s=.3)
plt.scatter(df['temp'],df['std_pixel_209_77'],s=.3, label='209, 77')
plt.scatter(df['temp'],df['std_pixel_237_44'],s=.3, label='237, 44')
plt.scatter(df['temp'],df['std_pixel_278_33'],s=.3, label='278, 33')
plt.scatter(df['temp'],df['std_pixel_284_49'],s=.3, label='284, 49')
plt.scatter(df['temp'],df['std_pixel_314_9'],s=.3, label='314, 9')
plt.legend()
plt.ylim(0,25)